# Stage 2 Notebook 57 - Exp2BBB Anchor + VFL + full 70K + backbone-LR throttle (joint conflict fix)

**The diagnosed joint-conflict fix.** NB48/51/54 all hit val_map50 = 0 at full data despite lambda_det re-weighting (NB48: 1.0, NB51: 2.0, NB54: 3.0). Re-weighting the head losses doesn't fix the underlying conflict because BOTH losses backpropagate through the same shared backbone, and at full data the lane gradient norm dominates.

Diagnosis: the issue is in BACKBONE updates, not head weighting. With `backbone_lr_mult=0.1` (default), the backbone moves significantly each step under both head gradients. Lane wins the tug-of-war at full data scale.

Fix: `backbone_lr_mult: 0.1 -> 0.01`. Backbone effectively quasi-frozen (still trains, just 10x slower than heads). Both tasks adapt to the slowly-moving backbone rather than fighting over it. This is the standard DETR-style 'lr_backbone=1e-6' recipe.

Diffs vs NB54 (Exp2YY exp49):
- `backbone_lr_mult: 0.1 -> 0.01` (the only critical change)
- `lambda_det: 3.0 -> 1.5` (gentler since we expect the backbone fix to fix det)
- `lambda_lane: 0.5 -> 1.0` (don't suppress lane)
- matcher: `topk_fixed -> dynamic_k` (preserve NB48's geometry champion)

### Run mode
1. `DEBUG_MODE=True` smoke.
2. `DEBUG_MODE=False` 6 epochs full data. ~60-80 min.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint_smoke.log
OK exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=4.6163 det_loss=3.7351 grad_cos=0.2548 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.5001453161239624, 'gate/lane_mean': 0.4997684955596924, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'full6'
    EPOCHS = 6
    BATCH_SIZE = 8
    LIMIT_TRAIN = None
    LIMIT_VAL = 2000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: None
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint_full6 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint_full6.tar --epochs 6 --batch-size 8 --limit-val 2000 --force-extract --print-every 50
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint_full6.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp52_rmt_gca_anchor_vfl_full_data_bb_throttle_joint_full6_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp52_rmt_gca_anchor_vfl_full

0

## What to watch in Exp2BBB

Reference NB48: matched_iou=0.544, decoded_f1=0.050, val_det=3.16, val_map50=0.
Reference NB54: matched_iou=0.289, decoded_f1=0.062, val_det=3.10, val_map50=0.

Pass criteria at epoch 6:
- **`val_det <= 2.5` and `val/det/map50 >= 0.005`** -- the smoking gun. If backbone throttling fixes det, this hits.
- `val/matched_line_iou >= 0.45` -- accept a small regression from NB48's 0.544 (backbone moves less, so lane head trains slightly less geometry).
- `val/lane/decoded_f1 >= 0.04` -- maintain NB48 level.
- `train/grad_cosine_epoch_mean >= 0` -- diagnostic confirming the conflict is reduced.